# Arbeiten mit Klimadaten I: die Säkularstation in Potsdam

## Eure erste Amtshandlung in Python

In [ ]:
# Hallo Python-Welt
print("Hallo Python. Goodbye Excel.")

## Lade benötigte Pakete

In [ ]:
import pandas as pd

## Lade den Datensatz: über 130 Jahre Potsdamer Klimageschichte

Bevor Ihr den Code ausführt: Navigiert auf der linken Seite einmal in das Verzeichnis `../../data` und schaut Euch die Datei `klima-potsdam.txt` an. 

In [ ]:
# Datei einlesen
df = pd.read_csv("../../data/klima-potsdam.txt", # ../../ heißt zwei Ebenen über dem Arbeitsverzeichnis
                 sep=";",                        # Spaltentrenner
                 skipinitialspace=True,          # whitespace aus den Spaltennamen entfernen
                 na_values="-999", )             # Das sind Fehlwerte
# Spalte "eor" rausschmeißen
df = df.drop(columns=["eor"])
# Spalte MESS_DATUM als Datum interpretieren
df["MESS_DATUM"] = pd.to_datetime(df.MESS_DATUM, format="%Y%m%d")
# MESS_DATUM als Zeilenindex verwenden
df = df.set_index("MESS_DATUM")

In [ ]:
# Und so sieht unser Datensatz aus (head and tail) - der Datentyp heißt "dataframe"
df

## Was bedeuten die Spalten?

Diese Info erhalten wir aus den Metadaten, die vom DWD bereitgestellt werden.

- `FM`: Tagesmittel der Windgeschwindigkeit, m/s
- `FX`: Maximum der Windspitze Messnetz 3, m/sec
- `NM`: Tagesmittel des Bedeckungsgrades, Achtel
- `PM`: Tagesmittel des Luftdrucks, hpa
- `RSK`: tgl. Niederschlagshoehe, mm
- `RSKF`: tgl. Niederschlagsform
- `SDK`: Sonnenscheindauer Tagessumme, Stunden
- `SHK_TAG`: Schneehoehe Tageswert, cm
- `TGK`: Minimum der Lufttemperatur am Erdboden in 5cm, Grad C
- `TMK`: Tagesmittel der Temperatur, Grad C
- `TNK`: Tagesminimum der Lufttemperatur in 2m Hoehe, Grad C
- `TXK`: Tagesmaximum der Lufttemperatur in 2m Hoehe, Grad C
- `UPM`: Tagesmittel der Relativen Feuchte, Prozent
- `VPM`: Tagesmittel des Dampfdruckes, hpa

### Einfach mal das Tagesmittel der Lufttemperatur plotten

In [ ]:
df.TMK.plot()

Hm...das ist noch nicht besonders aussagekräftig.

## Beobachteter Klimawandel in Potsdam

Oder, genauer gesagt, "Klimaerwärmung in Potsdam" (Klimawandel ist nicht nur Erwärmung).

In [ ]:
# Zeitreihe der Jahresmitteltemperatur
dfamean = df.resample("YE").mean()
dfasum = df.resample("YE").sum()

In [ ]:
dfamean.TMK.plot()

Was sehen wir? Warum ist 2026 so kalt?

### Aufgaben

Versuche nun mit Bastelei, Recherche und vor allem mit Hilfe von LLM (z.B. https://gptup.uni-potsdam.de/) folgende Aufgaben zu lösen.

1. Berechne den Mittelwert der Lufttemperatur in Potsdam für die gesamte Messreihe.
2. Berechne den Mittelwert der Lufttemperatur in Potsdam für die Klimanormalperioden 1931-1960, 1961-1990 und 1991-2020.
3. Verschönere die obige Abbildung: Füge eine y-Achsen-Beschriftung hinzu, entferne die x-Achsenbeschriftung, füge ein Gitter (grid) hinzu, füge horizontale Linien für die Werte in den Klimanormalperdioden hinzu.
4. Erstelle ähnliche Abbildungen für die mittlere Windgeschwindigkeit und die Jahresniederschlagssummme. Interpretation der Ergebnisse?
5. Wieviele Tage mit einer Mitteltemperatur von über 25 Grad Celsius gab es im Mittel pro Jahr in den Klimanormalperioden 1961-1990 und 1991-2020? Interpretation?
6. Wieviele Tage mit einer Schneehöhe von über 10 cm gab es im Mittel pro Jahr in den Klimanormalperioden 1961-1990 und 1991-2020? Interpretation?
7. Erstelle eine Abbildung des mittleren Jahresgangs der Lufttemperatur für die Klimanormalperioden 1961-1990 und 1991-2020.

In [ ]:
# Los geht's ...

## Wo gibt es weitere Klimastationen?

Der oben analysierte Datensatz stammt aus den [Open Data Repository des Deutschen Wetterdienstes](https://opendata.dwd.de/climate_environment/CDC/) (DWD). Ein unfassbarer Datenschatz, frei verfügbar für alle. Wir wollen nun darstellen, wo es weitere tägliche Klimastationsdaten gibt. Dafür schauen wir uns [diese Datei](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/KL_Tageswerte_Beschreibung_Stationen.txt) an.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import folium

In [ ]:
quelle = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/KL_Tageswerte_Beschreibung_Stationen.txt"
stationen = pd.read_fwf(quelle, encoding="ISO-8859-1", skiprows=2,
                        names=["id", "from", "to", "elevation", "lat", "lon", "name","state","abgabe"])
stationen

In [ ]:
gdf = gpd.GeoDataFrame(stationen, 
                       geometry=gpd.points_from_xy(stationen.lon, stationen.lat), crs="EPSG:4326")

In [ ]:
m = folium.Map(location=[51.4, 11.5], zoom_start=6, tiles="CartoDB positron")

for _, row in stationen.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,#row["count"] / 2,
        color="black",#species_colours.get(row["species"], "grey"),
        fill=True,
        fill_opacity=0.5,
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>"
            f"Elevation: {row['elevation']} m<br>"
            f"ID: {row['id']}<br>",
            max_width=200,
        ),
    ).add_to(m)

m